## Notebook 04: Clustering Behavior 

This clustering analysis explores whether listening events form distinct time-based usage patterns using hour-of-day and weekday features. Content and engagement dimensions are intentionally excluded due to data availability.

In [5]:
import sys
sys.executable



'C:\\Users\\trist\\PycharmProjects\\spotify-listening-analytics\\.venv\\Scripts\\python.exe'

In [4]:
import pandas as pd
from pathlib import Path

In [6]:
DATA_PATH = Path("../data/cleaned/listening_cleaned.parquet")
DATA_PATH.exists()

True

1. Goal & unit of clustering

Each row is a listening event.
I’m clustering events to discover listening modes.
clusters are interpretable and stable enough to describe in words


In [8]:
df = pd.read_parquet(DATA_PATH)
df.shape

(66098, 9)

In [9]:
df['timestamp_local'].dt.weekday.value_counts().sort_index()


timestamp_local
0     9435
1     9142
2     9043
3     8835
4    10392
5    10151
6     9100
Name: count, dtype: int64

In [10]:
df['timestamp_local'].dt.hour.value_counts().sort_index()

timestamp_local
0     1523
1      936
2      683
3      456
4      261
5      175
6      153
7     1247
8     1186
9     2025
10    2760
11    3422
12    3953
13    4220
14    4348
15    4813
16    4807
17    5643
18    5397
19    4503
20    3590
21    3659
22    3593
23    2745
Name: count, dtype: int64

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [14]:
scaler = StandardScaler()


In [17]:
df['hour'] = df['timestamp_local'].dt.hour
df['day'] = df['timestamp_local'].dt.weekday

In [25]:
x = df[['hour', 'day']]

In [29]:
x_scaled = scaler.fit_transform(x)

In [32]:
x_scaled.shape



(66098, 2)

In [37]:
kmeans = KMeans(n_clusters=5)
labels = kmeans.fit_predict(x_scaled)


In [39]:
len(labels)

66098

In [40]:
len(x_scaled)

66098

In [42]:
import numpy as np


In [43]:
np.unique(labels)


array([0, 1, 2, 3, 4], dtype=int32)

In [44]:
df['cluster'] = labels


In [45]:
df[['hour', 'day', 'cluster']].head()


,hour,day,cluster
0,15,4,1
1,15,4,1
2,15,4,1
3,15,4,1
4,15,4,1


In [47]:
#learning groupby()
grouped_data = df[['hour', 'day', 'cluster']].groupby('cluster')['hour'].mean()

In [48]:
grouped_data.head()

cluster
0    11.723930
1    12.417315
2    19.029502
3    19.058166
4     1.839223
Name: hour, dtype: float64

In [56]:
#clusters per day of the week 

clusters_perday = (
    df[['hour', 'day', 'cluster']]
    .groupby(['cluster', 'day'])['cluster']
    .count()
)
clusters_perday.head()

cluster  day
0        0      4323
         1      3798
         2      3712
         3      3453
1        4      4569
Name: cluster, dtype: int64

In [63]:
counts = (
    df
    .groupby(['cluster', 'day']) 
    .size()
)
    
counts.head()

cluster  day
0        0      4323
         1      3798
         2      3712
         3      3453
1        4      4569
dtype: int64

In [65]:
cluster_totals = (
    df
    .groupby('cluster')
    .size()
)

cluster_totals.head()

cluster
0    15286
1    12602
2    14338
3    19599
4     4273
dtype: int64

In [70]:
cluster_day_proportions = (
    counts 
    / cluster_totals
)

cluster_day_proportions = cluster_day_proportions.unstack()
cluster_day_proportions.head()

day,0,1,2,3,4,5,6
cluster,,,,,,,
0,0.282808,0.248463,0.242837,0.225893,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,0.362561,0.344707,0.292731
2,NaN,NaN,NaN,NaN,0.363719,0.346910,0.289371
3,0.247053,0.260472,0.247309,0.245166,NaN,NaN,NaN
4,0.063187,0.055933,0.113269,0.135034,0.142289,0.194945,0.295343
